# 28 — Demo: topology-aware federated AMR prediction (final protocol)

Same pipeline as notebooks 16 (Louvain, 3 hospitals) and 17 (organism split, 10 hospitals), merged into one notebook. The only difference is that **the results were checkpointed**, so every arm is skipped (`skip <arm>: already saved`) and nothing is trained. The tables and figures below are computed live from those results.

**Protocol:** each hospital picks its own cutoff and the round on its validation rows; test is read once. Primary metric: test macro-F1, size-weighted. 5 seeds (42–46), 8 rounds × 4 local epochs, hidden 128.

In [ ]:
# 1) Environment (local; needs ARMD_DIR for the cohort)
import os, sys, glob, json, warnings
from pathlib import Path
import numpy as np
warnings.filterwarnings('ignore')
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
if 'ARMD_DIR' not in os.environ:
    hits = glob.glob(os.path.expanduser('~/Library/CloudStorage/GoogleDrive-*/My Drive/ARMD'))
    if hits:
        os.environ['ARMD_DIR'] = hits[0]
from amr_fed import config
D = Path(config.DATA_DIR)
print('repo:', ROOT, '| DATA_DIR:', D, '| exists:', D.exists())
assert D.exists(), 'ARMD_DIR is wrong.'
from amr_fed.federated import final_protocol as P, analysis as A
print('imports OK | v3 temperatures:', P.T_GRID, '| fine-tune epochs:', P.FT_EPOCHS)

## The data
ARMD, filtered to positive cultures with Susceptible / Intermediate / Resistant labels. Hospitals are simulated by splitting **patients** (1:1, no leakage).

In [ ]:
# 2) Load the cohort once
from amr_fed.data_loader import load_cohort_frame, PK
df = load_cohort_frame()
print(f"{len(df):,} susceptibility results, {df[PK].nunique():,} patients, resistant rate {df['label'].mean():.1%}")
RES = {}

## Louvain split (3 hospitals)
Organism–antibiotic communities found with Louvain, packed into 3 hospitals. The partition changes with the seed.

In [ ]:
# louvain: partition check on the real cohort
from amr_fed.partition import louvain_split, partition_summary
SEEDS = (42, 43, 44, 45, 46)
for s in (SEEDS if 'louvain' == 'louvain' else SEEDS[:1]):
    a = louvain_split(df, n_clients=3, seed=s)
    print(f'seed {s}: {a.nunique()} hospitals | patients (largest first): {a.value_counts().sort_values(ascending=False).tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'

In [ ]:
# louvain: run every arm (resume-safe). Saved results exist, so every arm is skipped and nothing is trained.
os.environ.pop('AMR_RNG_PROBE', None)
import ray
TAG = 'all'
KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: louvain_split(d, n_clients=3, seed=s),
          n_clients=3, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RESULTS = ROOT / 'results' / f'amr_louvain_final_{TAG}.json'
res = json.load(open(RESULTS))
specs = [k for k, _, _ in P.stage1_specs('louvain', False)] + ['v3_shuffled']
missing = [k for k in specs if k not in res]
assert not missing, f'results file lacks {missing} - this demo must not train; stop here'
res = P.run_all(res, prefix='louvain', include_trimmed=False, kw=KW, ray_shutdown=ray.shutdown)   # no checkpoint: file is never rewritten
RES['louvain'] = res
print('arms done:', [k for k in res if not k.startswith('_')])

In [ ]:
# louvain: summary, pre-registered verdict, paired comparisons
res = RES['louvain']
print(A.summary_table(res))
print('\nchosen v3 temperature (validation only):', res['_meta'].get('chosen_T'), '| mean val F1 by T:', res['_meta'].get('T_val_scores'))
print('wall-clock per arm of the original run (min, mean over seeds):', {k: round(np.mean([r["wall_seconds"] for r in v["runs"]]) / 60, 1) for k, v in res.items() if not k.startswith('_')})
print()
print(A.verdict(res))
print(A.comparison_report(res))

In [ ]:
# louvain: per-hospital test F1 and the aggregation weights
print(A.per_hospital_table(res, rank_by_size=True))
print(A.weights_report(res))

## Organism split (10 hospitals)
Each patient goes to the hospital holding their most common organism. One fixed partition for all seeds; one hospital holds about 57% of the patients.

In [ ]:
# org10: partition check on the real cohort
from amr_fed.partition import organism_community, partition_summary
SEEDS = (42, 43, 44, 45, 46)
for s in (SEEDS if 'org10' == 'louvain' else SEEDS[:1]):
    a = organism_community(df, n_clients=10, seed=s)
    print(f'seed {s}: {a.nunique()} hospitals | patients (largest first): {a.value_counts().sort_values(ascending=False).tolist()}')
    assert a.index.is_unique and len(a) == df[PK].nunique(), 'not 1:1'

In [ ]:
# org10: run every arm (resume-safe). Saved results exist, so every arm is skipped and nothing is trained.
os.environ.pop('AMR_RNG_PROBE', None)
import ray
TAG = 'all'
KW = dict(seeds=SEEDS, df=df, partition_fn=lambda d, s: organism_community(d, n_clients=10, seed=s),
          n_clients=10, rounds=8, local_epochs=4, local_only_epochs=32, hidden=128)
RESULTS = ROOT / 'results' / f'amr_org10_final_{TAG}.json'
res = json.load(open(RESULTS))
specs = [k for k, _, _ in P.stage1_specs('org10', False)] + ['v3_shuffled']
missing = [k for k in specs if k not in res]
assert not missing, f'results file lacks {missing} - this demo must not train; stop here'
res = P.run_all(res, prefix='org10', include_trimmed=False, kw=KW, ray_shutdown=ray.shutdown)   # no checkpoint: file is never rewritten
RES['org10'] = res
print('arms done:', [k for k in res if not k.startswith('_')])

In [ ]:
# org10: summary, pre-registered verdict, paired comparisons
res = RES['org10']
print(A.summary_table(res))
print('\nchosen v3 temperature (validation only):', res['_meta'].get('chosen_T'), '| mean val F1 by T:', res['_meta'].get('T_val_scores'))
print('wall-clock per arm of the original run (min, mean over seeds):', {k: round(np.mean([r["wall_seconds"] for r in v["runs"]]) / 60, 1) for k, v in res.items() if not k.startswith('_')})
print()
print(A.verdict(res))
print(A.comparison_report(res))

In [ ]:
# org10: per-hospital test F1 and the aggregation weights
print(A.per_hospital_table(res, rank_by_size=False))
print(A.weights_report(res))

## Figures
Computed from the loaded results (same code as the slide figures).

In [ ]:
# 3) Figure 1: every arm on both splits (dots = seeds, marker = mean ± sd)
import matplotlib.pyplot as plt
BLUE, ORANGE, AQUA, GRAY = '#2a78d6', '#eb6834', '#1baf7a', '#8a8985'
INK, INK2, GRID, SURF = '#0b0b0b', '#52514e', '#e4e3df', '#fcfcfb'
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': GRID, 'axes.labelcolor': INK2, 'xtick.color': INK2,
    'ytick.color': INK2, 'text.color': INK, 'axes.facecolor': SURF, 'figure.facecolor': SURF,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.color': GRID,
    'grid.linewidth': 0.6, 'axes.axisbelow': True})
SPLITS = {'louvain': 'Louvain split (3 hospitals)', 'org10': 'Organism split (10 hospitals)'}
ARMS = [('local', 'Local-only', GRAY), ('fedavg', 'FedAvg', BLUE), ('v3', 'Topo personalized', ORANGE),
        ('uniform', 'Uniform weights', '#9b9a96'), ('topo', 'Topo', '#6b6a66'),
        ('fedprox', 'FedProx', '#b8c7e0'), ('fedmedian', 'FedMedian', '#c9c8c3')]
def seeds_of(d, arm, key):
    return np.array([r[key] for r in d[arm]['runs']], float)
def arm_scores(d, key):
    if key == 'local':
        return seeds_of(d, 'fedavg', 'val_local')
    if key == 'v3':
        return seeds_of(d, f"v3_T{d['_meta']['chosen_T']:g}", 'phc_f1')
    return seeds_of(d, key, 'phc_f1')
SC = {s: {k: arm_scores(RES[s], k) for k, _, _ in ARMS} for s in SPLITS}
fig, axs = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, s in zip(axs, SPLITS):
    order = list(range(len(ARMS)))[::-1]
    for y, i in zip(range(len(ARMS)), order):
        k, lab, col = ARMS[i]
        v = SC[s][k]
        ax.scatter(v, [y] * len(v), s=14, color=col, alpha=0.45, zorder=2, linewidths=0)
        ax.errorbar(v.mean(), y, xerr=v.std(), fmt='o', color=col, ms=7, mec=SURF, mew=1.5, capsize=3, lw=1.5, zorder=3)
    ax.set_yticks(range(len(ARMS))); ax.set_yticklabels([ARMS[i][1] for i in order])
    ax.set_title(SPLITS[s], loc='left', fontsize=11, color=INK)
    ax.set_xlabel('Test macro-F1 (dots = 5 seeds, marker = mean ± sd)')
    ax.xaxis.set_major_locator(plt.MaxNLocator(5)); ax.grid(axis='y', visible=False)
fig.suptitle('Federated vs local-only on two different hospital partitions', x=0.01, ha='left', fontsize=13)
fig.tight_layout(); plt.show()

In [ ]:
# 4) Figure 2: paired gain over local-only (bars = mean, whiskers = 95% CI, dots = seeds)
T975 = 2.776  # t critical, 95% two-sided, df = 4
def ci(x):
    return x.mean(), T975 * x.std(ddof=1) / np.sqrt(len(x))
gaps = [('fedavg', 'FedAvg', BLUE), ('v3', 'Topo\npersonalized', ORANGE), ('uniform', 'Uniform', GRAY)]
fig, axs = plt.subplots(1, 2, figsize=(11, 3.9), sharey=True)
for ax, s in zip(axs, SPLITS):
    for x, (k, lab, col) in enumerate(gaps):
        diff = SC[s][k] - SC[s]['local']
        m, h = ci(diff)
        ax.bar(x, m, color=col, width=0.55)
        ax.errorbar(x, m, yerr=h, color=INK, capsize=4, lw=1.4)
        ax.scatter([x] * len(diff), diff, s=12, color=INK, alpha=0.4, zorder=3, linewidths=0)
        ax.text(x, max(m + h, diff.max()) + 0.0007, f'{m:+.4f}', ha='center', fontsize=9, color=INK2,
                bbox=dict(fc=SURF, ec='none', pad=1.5), zorder=5)
    ax.axhline(0, color=INK2, lw=1); ax.set_ylim(-0.0055, 0.0135)
    ax.set_xticks(range(len(gaps))); ax.set_xticklabels([g[1] for g in gaps])
    ax.set_title(SPLITS[s], loc='left', fontsize=11, color=INK); ax.grid(axis='x', visible=False)
axs[0].set_ylabel('F1 gain over local-only\n(paired by seed, 95% CI)')
fig.suptitle("Federation's gain over local-only: small when hospitals have plenty of data", x=0.01, ha='left', fontsize=13)
fig.tight_layout(); plt.show()

## How the gap changes with the amount of data
Each hospital keeps only a fraction of its train/validation patients (test patients unchanged). Gap = federated F1 minus local-only F1, paired by seed. Saved results from the low-data runs; the 100% point is the final-protocol run above.

**Read with care:** the low-data runs use 3 seeds (42–44) against 5 at 100%, and local-only trains 40 epochs there against 32 at 100%. With little data every epoch is one tiny full-batch step, so extra epochs matter. Organism split only goes down to 0.3%, Louvain only to 3%.

In [ ]:
# 5) Figure 3: gain over local-only vs fraction of train data kept (dots = seeds, marker = mean ± sd)
LOW = {'louvain': ('louvain3', [0.03, 0.10]), 'org10': ('org10', [0.003, 0.01, 0.03, 0.10])}
def lowfile(tag, x):
    stem = {0.003: 'f003m', 0.01: 'f001', 0.03: 'f003', 0.10: 'f010'}[x]
    return ROOT / 'results' / f'amr_lowdata_all_{tag}_{stem}.json'
LOWARMS = [('fedavg', 'FedAvg', BLUE), ('v3_topo', 'Topo personalized', ORANGE)]
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, s in zip(axs, SPLITS):
    tag, levels = LOW[s]
    xs = {}
    for k, lab, col in LOWARMS:
        pts = []
        for x in levels:
            d = json.load(open(lowfile(tag, x)))
            loc = np.array([r['val_local'] for r in d['fedavg']['runs']], float)
            pts.append((x, np.array([r['phc_f1'] for r in d[k]['runs']], float) - loc))
        full = SC[s]['fedavg' if k == 'fedavg' else 'v3'] - SC[s]['local']
        pts.append((1.0, full))
        for x, v in pts:
            ax.scatter([x] * len(v), v, s=14, color=col, alpha=0.4, zorder=2, linewidths=0)
        ax.errorbar([p[0] for p in pts], [p[1].mean() for p in pts], yerr=[p[1].std() for p in pts],
                    fmt='-o', color=col, ms=6, mec=SURF, mew=1.2, capsize=3, lw=1.5, zorder=3, label=lab)
    ax.axhline(0, color=INK2, lw=1)
    ax.set_xscale('log')
    ticks = levels + [1.0]
    ax.set_xticks(ticks); ax.set_xticklabels([f'{t*100:g}%' for t in ticks])
    ax.minorticks_off()
    ax.set_title(SPLITS[s], loc='left', fontsize=11, color=INK)
    ax.set_xlabel('Share of each hospital\'s train data kept')
    ax.grid(axis='x', visible=False)
axs[0].set_ylabel('F1 gain over local-only\n(paired by seed)')
axs[0].legend(frameon=False, loc='best')
fig.suptitle('How federation\'s gain over local-only changes with the amount of data', x=0.01, ha='left', fontsize=13)
fig.tight_layout(); plt.show()